# 01 — Fragmentation Analysis with Super-Objects

Replicates **IndiFrag Tutorial 1** using open-source Python.

We intersect the LULC objects with district boundaries (super-objects),
then compute fragmentation metrics at three levels:

- **Object (O)** — each individual polygon
- **Class (Cl)** — all objects of the same land-use class within a district
- **Super-Object (SO)** — the district as a whole

This notebook covers the **Area and Perimeter** metric group.
Subsequent sections will add Shape, Aggregation, Diversity, and Contrast.

In [ ]:
import sys; sys.path.insert(0, "../src")

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from pathlib import Path

from indifrag.metrics.area_perimeter import (
    object_metrics, class_metrics, super_object_metrics
)

DATA_DIR = Path("../data/valencia")
TARGET_CRS = "EPSG:25830"

# Shared lookups
CLASS_NAMES = {
    1: "Agricultural", 2: "Barrenland", 3: "Commercial",
    4: "Green Urban Areas", 5: "Leisure Facilities",
    6: "Residential", 7: "Roads", 8: "Water",
}
CLASS_COLORS = {
    1: "#ffffbe", 2: "#e6a600", 3: "#ff8c00", 4: "#a3d774",
    5: "#267300", 6: "#c8374b", 7: "#cccccc", 8: "#73b2ff",
}
DISTRICT_NAMES = {
    "4625001": "Ciutat Vella", "4625002": "Eixample",
    "4625003": "Extramurs", "4625007": "L'Olivereta",
    "4625008": "Patraix", "4625009": "Jesús",
    "4625010": "Quatre Carreres",
}

## 1. Load and prepare data

In [ ]:
objects_raw = gpd.read_file(
    DATA_DIR / "ES003L2_VALENCIA_UA2006_Revised_Clipped_to_Core.shp"
).to_crs(TARGET_CRS)

districts = gpd.read_file(DATA_DIR / "VALENCIA_DISTR.shp").to_crs(TARGET_CRS)
districts["name"] = districts["CUDIS"].map(DISTRICT_NAMES)

centre = gpd.read_file(DATA_DIR / "VALENCIA_Centre_Point.shp").to_crs(TARGET_CRS)

print(f"Raw objects: {len(objects_raw)}")
print(f"Districts:   {len(districts)}")

## 2. Intersect objects with districts

IndiFrag clips every LULC polygon at district boundaries, so objects that
straddle two districts are split into separate pieces.  This is the key
difference from a simple centroid-based spatial join.

In [ ]:
objects_int = gpd.overlay(
    objects_raw,
    districts[["CUDIS", "name", "geometry"]],
    how="intersection",
)
objects_int["class_name"] = objects_int["CLASS"].map(CLASS_NAMES)

print(f"Objects after intersection: {len(objects_int)}  (was {len(objects_raw)})")
print(f"New fragments created:      {len(objects_int) - len(objects_raw)}")

## 3. Compute area and perimeter metrics

In [ ]:
# Object level
objects_int = object_metrics(objects_int)

# Class level
cl_metrics = class_metrics(objects_int, class_col="CLASS", so_col="CUDIS")
cl_metrics["class_name"] = cl_metrics["CLASS"].map(CLASS_NAMES)
cl_metrics["district"] = cl_metrics["SO"].map(DISTRICT_NAMES)

# Super-object level
so_metrics = super_object_metrics(objects_int, districts, class_col="CLASS", so_col="CUDIS")
so_metrics["district"] = so_metrics["CUDIS"].map(DISTRICT_NAMES)

print("Object-level metrics computed for", len(objects_int), "polygons")
print("Class-level metrics:", len(cl_metrics), "rows (class x district)")
print("Super-object metrics:", len(so_metrics), "districts")

---

## 4. Visualising the metrics

### 4.1 Object-level: area distribution by class

How large are individual LULC patches?  Residential parcels are typically
small and numerous, while agricultural areas are few but large.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: box plot of object areas by class
class_order = sorted(CLASS_NAMES.keys())
data_by_class = [
    objects_int.loc[objects_int["CLASS"] == c, "AreaO"].values * 1e6  # back to m²
    for c in class_order
]
bp = axes[0].boxplot(
    data_by_class, vert=True, patch_artist=True, showfliers=False,
    labels=[CLASS_NAMES[c] for c in class_order],
)
for patch, c in zip(bp["boxes"], class_order):
    patch.set_facecolor(CLASS_COLORS[c])
axes[0].set_ylabel("Object area (m²)")
axes[0].set_title("Object area distribution by class")
axes[0].tick_params(axis="x", rotation=45)

# Right: count of objects per class
counts = objects_int.groupby("CLASS").size()
bars = axes[1].bar(
    [CLASS_NAMES[c] for c in counts.index],
    counts.values,
    color=[CLASS_COLORS[c] for c in counts.index],
    edgecolor="#333333", linewidth=0.5,
)
axes[1].set_ylabel("Number of objects")
axes[1].set_title("Object count by class")
axes[1].tick_params(axis="x", rotation=45)
for bar, v in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 10, str(v),
                ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/figures/01_object_area_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.2 Class level: class density (DC) by district

**DC** = AreaCl / AreaSO — what proportion of each district is occupied by each class?

This is the land-use composition fingerprint of each district.

In [ ]:
# Pivot DC to a matrix: districts × classes
dc_pivot = cl_metrics.pivot_table(
    index="district", columns="class_name", values="DC", fill_value=0
)
# Order columns by class id
col_order = [CLASS_NAMES[c] for c in sorted(CLASS_NAMES.keys()) if CLASS_NAMES[c] in dc_pivot.columns]
dc_pivot = dc_pivot[col_order]

fig, ax = plt.subplots(figsize=(12, 5))
dc_pivot.plot.bar(
    stacked=True, ax=ax,
    color=[CLASS_COLORS[c] for c in sorted(CLASS_NAMES.keys()) if CLASS_NAMES[c] in dc_pivot.columns],
    edgecolor="white", linewidth=0.5,
)
ax.set_ylabel("Class density (DC)")
ax.set_title("Land-use composition by district — Class Density (DC)")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("../outputs/figures/01_class_density_stacked.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.3 Choropleth map: residential density per district

Map the class density of **Residential** (class 6) across the seven districts.
Darker = higher proportion of the district is residential.

In [ ]:
# Merge residential DC into districts GeoDataFrame
residential_dc = cl_metrics.loc[
    cl_metrics["CLASS"] == 6, ["SO", "DC"]
].rename(columns={"SO": "CUDIS", "DC": "DC_residential"})

districts_plot = districts.merge(residential_dc, on="CUDIS", how="left")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: LULC map
objects_int.plot(
    ax=axes[0],
    color=objects_int["CLASS"].map(CLASS_COLORS),
    edgecolor="#888888", linewidth=0.1,
)
districts.boundary.plot(ax=axes[0], color="black", linewidth=1.5)
axes[0].set_title("LULC classes", fontsize=12)
axes[0].set_axis_off()

# Right: Residential DC choropleth
districts_plot.plot(
    ax=axes[1], column="DC_residential",
    cmap="YlOrRd", edgecolor="black", linewidth=1.5,
    legend=True, legend_kwds={"label": "Residential density (DC)", "shrink": 0.6},
)
for _, row in districts_plot.iterrows():
    c = row.geometry.centroid
    axes[1].annotate(
        f"{row['name']}\n{row['DC_residential']:.1%}",
        xy=(c.x, c.y), ha="center", va="center", fontsize=8, fontweight="bold",
    )
axes[1].set_title("Residential class density (DC) by district", fontsize=12)
axes[1].set_axis_off()

plt.suptitle("Class Density — Residential", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/figures/01_residential_dc_map.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.4 Super-object level: district profiles

Compare districts on area, total perimeter, number of objects, and class count.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

so_sorted = so_metrics.sort_values("AreaSO")
bar_color = "#4a90d9"

# AreaSO
axes[0,0].barh(so_sorted["district"], so_sorted["AreaSO"], color=bar_color)
axes[0,0].set_xlabel("Area (km²)")
axes[0,0].set_title("District area (AreaSO)")
for i, v in enumerate(so_sorted["AreaSO"]):
    axes[0,0].text(v + 0.1, i, f"{v:.2f}", va="center", fontsize=8)

# PerimT
axes[0,1].barh(so_sorted["district"], so_sorted["PerimT"], color="#e07b39")
axes[0,1].set_xlabel("Total unique perimeter (km)")
axes[0,1].set_title("Total perimeter without duplicity (PerimT)")
for i, v in enumerate(so_sorted["PerimT"]):
    axes[0,1].text(v + 1, i, f"{v:.1f}", va="center", fontsize=8)

# NobSO
axes[1,0].barh(so_sorted["district"], so_sorted["NobSO"], color="#5bb370")
axes[1,0].set_xlabel("Number of objects")
axes[1,0].set_title("Object count (NobSO)")
for i, v in enumerate(so_sorted["NobSO"]):
    axes[1,0].text(v + 3, i, str(int(v)), va="center", fontsize=8)

# NCl
axes[1,1].barh(so_sorted["district"], so_sorted["NCl"], color="#9b59b6")
axes[1,1].set_xlabel("Number of classes")
axes[1,1].set_title("Class count (NCl)")
axes[1,1].set_xlim(0, 9)
for i, v in enumerate(so_sorted["NCl"]):
    axes[1,1].text(v + 0.1, i, str(int(v)), va="center", fontsize=8)

plt.suptitle("Super-Object Level Metrics", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/figures/01_super_object_profiles.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 5. Validation against IndiFrag

Compare our computed values with the expected results from IndiFrag.
Small differences are expected due to geometric precision in the overlay operation.

In [ ]:
# Load expected results
exp_so = pd.read_csv("../reference/expected_results/Results_FI_SO.txt", sep="\t", skiprows=4)
exp_so.columns = exp_so.columns.str.strip()
exp_so["CUDIS"] = exp_so["CUDIS"].astype(str)
exp_so["district"] = exp_so["CUDIS"].map(DISTRICT_NAMES)

exp_cl = pd.read_csv(
    "../reference/expected_results/Results_FI_Cls.txt",
    sep="\t", skiprows=4,
)
exp_cl.columns = exp_cl.columns.str.strip()
exp_cl["SO"] = exp_cl["SO"].astype(str)
exp_cl.rename(columns={"CLASE": "CLASS"}, inplace=True)

### 5.1 Super-object level validation

In [ ]:
comp_so = so_metrics.merge(exp_so, on="CUDIS", suffixes=("_ours", "_exp"))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, label in zip(axes, ["AreaSO", "PerimT", "NobSO"],
                                ["Area (km²)", "Total Perimeter (km)", "Object Count"]):
    ours = comp_so[f"{col}_ours"]
    exp = comp_so[f"{col}_exp"]
    lo = min(ours.min(), exp.min()) * 0.9
    hi = max(ours.max(), exp.max()) * 1.1
    ax.plot([lo, hi], [lo, hi], "--", color="#999999", linewidth=1, label="1:1 line")
    ax.scatter(exp, ours, s=60, color="#c8374b", zorder=3)
    for _, r in comp_so.iterrows():
        ax.annotate(r["district_ours"], (r[f"{col}_exp"], r[f"{col}_ours"]),
                    fontsize=7, xytext=(5, 5), textcoords="offset points")
    ax.set_xlabel(f"IndiFrag {label}")
    ax.set_ylabel(f"fragmetrics {label}")
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle("Validation: fragmetrics vs IndiFrag (Super-Object Level)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/01_validation_SO.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Class level validation

In [ ]:
comp_cl = cl_metrics.merge(
    exp_cl[["CLASS", "SO", "AreaCl", "PerimCl", "NobCl", "DC"]],
    on=["CLASS", "SO"], suffixes=("_ours", "_exp")
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, label in zip(axes, ["AreaCl", "DC", "NobCl"],
                                ["Class Area (km²)", "Class Density (DC)", "Object Count"]):
    ours = comp_cl[f"{col}_ours"]
    exp = comp_cl[f"{col}_exp"]
    lo = min(ours.min(), exp.min()) * 0.9
    hi = max(ours.max(), exp.max()) * 1.1
    if lo == 0: lo = -hi * 0.02
    ax.plot([lo, hi], [lo, hi], "--", color="#999999", linewidth=1, label="1:1 line")
    scatter = ax.scatter(
        exp, ours, s=30, alpha=0.7,
        c=[CLASS_COLORS.get(c, "grey") for c in comp_cl["CLASS"]],
        edgecolors="#333333", linewidth=0.3, zorder=3,
    )
    ax.set_xlabel(f"IndiFrag {label}")
    ax.set_ylabel(f"fragmetrics {label}")
    ax.set_title(col)
    ax.legend(fontsize=8)

# Legend for classes
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CLASS_COLORS[c],
           markersize=7, label=CLASS_NAMES[c])
    for c in sorted(CLASS_COLORS.keys())
]
fig.legend(handles=legend_elements, loc="lower center", ncol=8, fontsize=8,
           bbox_to_anchor=(0.5, -0.06))

plt.suptitle("Validation: fragmetrics vs IndiFrag (Class Level)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/01_validation_class.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Summary table

In [ ]:
summary = comp_so[["district_ours", "AreaSO_ours", "AreaSO_exp",
                    "PerimT_ours", "PerimT_exp",
                    "NobSO_ours", "NobSO_exp"]].copy()
summary.columns = ["District", "Area (ours)", "Area (IndiFrag)",
                    "PerimT (ours)", "PerimT (IndiFrag)",
                    "Nob (ours)", "Nob (IndiFrag)"]
summary["Area (ours)"] = summary["Area (ours)"].round(4)
summary["PerimT (ours)"] = summary["PerimT (ours)"].round(2)
summary["Nob (ours)"] = summary["Nob (ours)"].astype(int)
summary["Nob (IndiFrag)"] = summary["Nob (IndiFrag)"].astype(int)
summary

> **Note on small differences:** IndiFrag uses ArcGIS geoprocessing for the
> overlay intersection, which handles geometric precision differently from
> GeoPandas/Shapely.  This produces slightly different polygon splits at
> district boundaries, leading to minor variations in object counts and
> perimeters.  Area values match to 4 decimal places.

---

**Next:** Shape metrics (Fractal Dimension, Shape Index, Perimeter-Area Mean Ratio)
will be added in the next iteration of this notebook.